# 学習済みの翻訳モデルを自分の分野のデータでファインチューニングする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_machine_translation/mt_finetune.ipynb)

NLP クックブック 22-05 ／ 機械翻訳

> GPU ランタイム推奨 (ランタイム → ランタイムのタイプを変更 → T4 GPU)

公開されている翻訳モデルを、自分の分野 (ここでは映画やドラマの字幕) の対訳データで追加学習して、その分野での訳質を上げたい。

In [ ]:
!pip install -q datasets sentencepiece sacremoses sacrebleu accelerate

In [ ]:
import re
from datasets import Dataset, load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "Helsinki-NLP/opus-mt-ja-en"        # 日英の Marian モデル (約 300MB)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

ds = load_dataset("Helsinki-NLP/opus-100", "en-ja")
def clean(pairs):
    return [p for p in pairs if 0 < len(p["ja"]) <= 80 and 0 < len(p["en"].split()) <= 40 and re.search(r"[ぁ-んァ-ヶ一-龥]", p["ja"])]

train = Dataset.from_list(clean(ds["train"]["translation"][:60000])).shuffle(seed=0).select(range(30000))
test = Dataset.from_list(clean(ds["test"]["translation"])).select(range(500))

def preprocess(batch):
    return tokenizer(batch["ja"], text_target=batch["en"], truncation=True, max_length=64)   # text_target が訳文側 (labels になる)

train_enc = train.map(preprocess, batched=True, remove_columns=["ja", "en"])
test_enc = test.map(preprocess, batched=True, remove_columns=["ja", "en"])

In [ ]:
import sacrebleu
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def translate(model, sentences, batch_size=64):
    model.to(device).eval()
    out = []
    for i in range(0, len(sentences), batch_size):
        batch = tokenizer(sentences[i:i + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=64).to(device)
        out += tokenizer.batch_decode(model.generate(**batch, num_beams=4, max_new_tokens=64), skip_special_tokens=True)
    return out

before = translate(model, test["ja"])
print("微調整前:", sacrebleu.corpus_bleu(before, [test["en"]]))

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir="mt-finetuned", num_train_epochs=1, learning_rate=3e-5, warmup_steps=100, weight_decay=0.01,
    per_device_train_batch_size=32, fp16=True, logging_steps=100, save_strategy="no", report_to="none", seed=0,
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_enc, data_collator=DataCollatorForSeq2Seq(tokenizer, model=model))
trainer.train()

In [ ]:
after = translate(model, test["ja"])
print("微調整前:", sacrebleu.corpus_bleu(before, [test["en"]]))
print("微調整後:", sacrebleu.corpus_bleu(after, [test["en"]]))
for ja, en, b, a in list(zip(test["ja"], test["en"], before, after))[:6]:
    print(f"{ja}\n   前: {b}\n   後: {a}\n   参照: {en}")

model.save_pretrained("mt-finetuned"); tokenizer.save_pretrained("mt-finetuned")

- ゼロから学習するよりはるかに少ないデータと時間で、特定の分野の文体や用語に合わせられる。T4 GPU で 3 万文対・1 エポックが 5 分程度。
- 学習率は小さめ (1e-5〜5e-5) にする。大きすぎると、元のモデルが持っていた一般的な翻訳能力が壊れる (破滅的忘却)。
- 微調整後は、対象の分野だけでなく、一般的な文のテストセットでも評価する。分野内で上がって分野外で下がるのが典型的な副作用。
- BLEU の差が小さいときは、評価の章の検定 (ブートストラップ) で有意かどうかを確かめる。訳文を実際に見比べることも欠かせない。
- NLLB のような多言語モデルや、LLM を LoRA で微調整する方法 (LLM の章) でも同じ考え方が使える。